# Ground — valutazione activity recognition

Eseguire le celle in ordine in Colab (CPU sufficiente) oppure in Jupyter locale.
Il notebook contiene il valutatore e una copia dei 247 segmenti GT forniti: non occorre clonare il repository né caricare i file XLSM.

Una sola esecuzione finale produce metriche per azione e per trace, medie macro/micro/pesate, analisi senza Drop/Idle/Hold, matrice di confusione, mapping persone e object-centric event log. Servono lo ZIP delle azioni e la cartella (o ZIP) delle associazioni. Nessuna GPU richiesta.

**Limiti:** identità stimate massimizzando l’accordo con le azioni GT per ogni clip; risultati ottimistici, non identità verificate. Allineamento assunto: 30 FPS, scene1 dall’inizio del video, blocchi da 300 frame, GT End incluso. Se preprocessing o annotazioni cambiano, verificare queste ipotesi prima di usare i risultati.


In [ ]:
from pathlib import Path
import shutil

try:
    from google.colab import drive
except ImportError:
    IN_COLAB = False
else:
    IN_COLAB = True
    drive.mount('/content/drive')

WORK_DIR = Path('/content/ground_evaluation') if IN_COLAB else Path.cwd() / 'ground_evaluation'
WORK_DIR.mkdir(parents=True, exist_ok=True)


## Percorsi degli input e dei risultati
Controllare PREDICTIONS_ZIP e ASSOCIATIONS_PATH (cartella o ZIP). GT_CSV=None usa i 247 segmenti incorporati.
OUTPUT_DIR è la cartella principale dei risultati su Drive: ogni esecuzione crea una sottocartella run distinta e il relativo ZIP, senza mescolare risultati precedenti.


In [ ]:
DRIVE_DATA = Path('/content/drive/MyDrive/AA-STAL/data_pipeline/DATA_ROOT')
PREDICTIONS_ZIP = DRIVE_DATA / 'ground_action_recognition_v1.zip' if IN_COLAB else Path('ground_action_recognition_v1.zip')
GT_CSV = None  # oppure Path('/percorso/ground_truth_actions.csv')
OUTPUT_DIR = DRIVE_DATA / 'ground_activity_evaluation_v1' if IN_COLAB else WORK_DIR / 'results'

ASSOCIATIONS_PATH = DRIVE_DATA / 'person_object_associations' if IN_COLAB else Path('person_object_associations')

if not PREDICTIONS_ZIP.is_file():
    raise FileNotFoundError(f'ZIP non trovato: {PREDICTIONS_ZIP}. Correggere PREDICTIONS_ZIP e rieseguire questa cella.')


## Ground truth
Copia dei dati estratti dai cinque file originali, senza conversioni di Start/End. Le intestazioni di persone prive di segmenti non sono esempi Idle.


In [ ]:
EMBEDDED_GT_CSV = 'source_file,sheet,row,person,verb,noun,collaborative_action,collaborative_person,start,end\nP000103_T1_V1.xlsm,P000103_T1_V1,5,Person B,Idle,,,,0,291\nP000103_T1_V1.xlsm,P000103_T1_V1,6,Person B,Grasp,Shaker,,,292,313\nP000103_T1_V1.xlsm,P000103_T1_V1,7,Person B,Hold,Shaker,Collaborative_Twist,with Person C,314,377\nP000103_T1_V1.xlsm,P000103_T1_V1,8,Person B,Hold,Shaker,,,378,438\nP000103_T1_V1.xlsm,P000103_T1_V1,9,Person B,Hold,Shaker,Collaborative_Drop,with Person C,439,473\nP000103_T1_V1.xlsm,P000103_T1_V1,10,Person B,Hold,Shaker,,,474,549\nP000103_T1_V1.xlsm,P000103_T1_V1,11,Person B,Hold,Shaker,Collaborative_Drop,with Person C,550,600\nP000103_T1_V1.xlsm,P000103_T1_V1,12,Person B,Hold,Shaker,,,601,655\nP000103_T1_V1.xlsm,P000103_T1_V1,13,Person B,Hold,Shaker,Collaborative_Drop,with Person C,656,694\nP000103_T1_V1.xlsm,P000103_T1_V1,14,Person B,Place_Down,Shaker,,,695,741\nP000103_T1_V1.xlsm,P000103_T1_V1,15,Person B,Handover,Muddler,Collaborative_Handover,with Person C,742,783\nP000103_T1_V1.xlsm,P000103_T1_V1,16,Person B,Squash,,,,784,1061\nP000103_T1_V1.xlsm,P000103_T1_V1,17,Person B,Place_Down,Muddler,,,1062,1107\nP000103_T1_V1.xlsm,P000103_T1_V1,18,Person B,Grasp,Alcohol_Bottle,,,1108,1135\nP000103_T1_V1.xlsm,P000103_T1_V1,19,Person B,Hold,Alcohol_Bottle,Collaborative_Twist,with Person C,1136,1261\nP000103_T1_V1.xlsm,P000103_T1_V1,20,Person B,Pour,Alcohol,,,1262,1378\nP000103_T1_V1.xlsm,P000103_T1_V1,21,Person B,Place_Down,Alcohol_Bottle,,,1379,1445\nP000103_T1_V1.xlsm,P000103_T1_V1,22,Person B,Grasp,Shaker,,,1446,1492\nP000103_T1_V1.xlsm,P000103_T1_V1,23,Person B,Hold,Shaker,Collaborative_Twist,with Person C,1493,1648\nP000103_T1_V1.xlsm,P000103_T1_V1,24,Person B,Hold,Shaker,,,1649,1670\nP000103_T1_V1.xlsm,P000103_T1_V1,25,Person B,Shake,Shaker,,,1671,1930\nP000103_T1_V1.xlsm,P000103_T1_V1,26,Person B,Hold,Shaker,Collaborative_Twist,with Person C,1931,2054\nP000103_T1_V1.xlsm,P000103_T1_V1,27,Person B,Pour,Cocktail,,,2055,2309\nP000103_T1_V1.xlsm,P000103_T1_V1,28,Person B,Place_Down,Shaker,,,2310,2344\nP000103_T1_V1.xlsm,P000103_T1_V1,29,Person B,Idle,,,,2345,2433\nP000103_T1_V1.xlsm,P000103_T1_V1,32,Person C,Idle,,,,0,145\nP000103_T1_V1.xlsm,P000103_T1_V1,33,Person C,Grasp,Knife,,,146,190\nP000103_T1_V1.xlsm,P000103_T1_V1,34,Person C,Cut,Lime,,,191,259\nP000103_T1_V1.xlsm,P000103_T1_V1,35,Person C,Place_Down,Knife,,,260,307\nP000103_T1_V1.xlsm,P000103_T1_V1,36,Person C,Twist,Shaker_Lid,Collaborative_Twist,with Person B,308,370\nP000103_T1_V1.xlsm,P000103_T1_V1,37,Person C,Place_Down,Shaker_Lid,,,371,410\nP000103_T1_V1.xlsm,P000103_T1_V1,38,Person C,Grasp,Sugar,,,411,440\nP000103_T1_V1.xlsm,P000103_T1_V1,39,Person C,Drop,Sugar,Collaborative_Drop,with Person B,441,478\nP000103_T1_V1.xlsm,P000103_T1_V1,40,Person C,Grasp,Lime,,,479,549\nP000103_T1_V1.xlsm,P000103_T1_V1,41,Person C,Drop,Lime,Collaborative_Drop,with Person B,550,616\nP000103_T1_V1.xlsm,P000103_T1_V1,42,Person C,Grasp,Ice,,,617,656\nP000103_T1_V1.xlsm,P000103_T1_V1,43,Person C,Drop,Ice,Collaborative_Drop,with Person B,657,708\nP000103_T1_V1.xlsm,P000103_T1_V1,44,Person C,Grasp,Muddler,,,709,740\nP000103_T1_V1.xlsm,P000103_T1_V1,45,Person C,Handover,Muddler,Collaborative_Handover,with Person B,741,794\nP000103_T1_V1.xlsm,P000103_T1_V1,46,Person C,Idle,,,,795,1131\nP000103_T1_V1.xlsm,P000103_T1_V1,47,Person C,Twist,Bottle_Cap,Collaborative_Twist,with Person B,1132,1252\nP000103_T1_V1.xlsm,P000103_T1_V1,48,Person C,Place_Down,Bottle_Cap,,,1253,1283\nP000103_T1_V1.xlsm,P000103_T1_V1,49,Person C,Idle,,,,1284,1463\nP000103_T1_V1.xlsm,P000103_T1_V1,50,Person C,Grasp,Shaker_Lid,,,1464,1492\nP000103_T1_V1.xlsm,P000103_T1_V1,51,Person C,Twist,Shaker_Lid,Collaborative_Twist,with Person B,1493,1649\nP000103_T1_V1.xlsm,P000103_T1_V1,52,Person C,Idle,,,,1650,1937\nP000103_T1_V1.xlsm,P000103_T1_V1,53,Person C,Twist,Shaker_Lid,Collaborative_Twist,with Person B,1938,2028\nP000103_T1_V1.xlsm,P000103_T1_V1,54,Person C,Place_Down,Shaker_Lid,,,2029,2056\nP000103_T1_V1.xlsm,P000103_T1_V1,55,Person C,Idle,,,,2057,2433\nP000103_T1_V2.xlsm,Annotations,5,Person B,Idle,,,,0,144\nP000103_T1_V2.xlsm,Annotations,6,Person B,Grasp,Knife,,,145,166\nP000103_T1_V2.xlsm,Annotations,7,Person B,Push,Lime,,,167,191\nP000103_T1_V2.xlsm,Annotations,8,Person B,Cut,Lime,,,192,253\nP000103_T1_V2.xlsm,Annotations,9,Person B,Place_Down,Knife,,,254,283\nP000103_T1_V2.xlsm,Annotations,10,Person B,Grasp,Lime,,,284,307\nP000103_T1_V2.xlsm,Annotations,11,Person B,Drop,Lime,Collaborative_Drop,with Person C,308,350\nP000103_T1_V2.xlsm,Annotations,12,Person B,Idle,,,,351,424\nP000103_T1_V2.xlsm,Annotations,13,Person B,Grasp,Muddler,,,425,455\nP000103_T1_V2.xlsm,Annotations,14,Person B,Handover,Muddler,Collaborative_Handover,with Person C,456,505\nP000103_T1_V2.xlsm,Annotations,15,Person B,Idle,,,,506,564\nP000103_T1_V2.xlsm,Annotations,16,Person B,Grasp,Alcohol_Bottle,,,565,594\nP000103_T1_V2.xlsm,Annotations,17,Person B,Twist,Bottle_Cap,,,595,673\nP000103_T1_V2.xlsm,Annotations,18,Person B,Place_Down,Bottle_Cap,,,674,696\nP000103_T1_V2.xlsm,Annotations,19,Person B,Hold,Alcohol_Bottle,,,697,727\nP000103_T1_V2.xlsm,Annotations,20,Person B,Pour,Alcohol,Collaborative_Pour,with Person C,728,978\nP000103_T1_V2.xlsm,Annotations,21,Person B,Grasp,Bottle_Cap,,,979,1001\nP000103_T1_V2.xlsm,Annotations,22,Person B,Twist,Bottle_Cap,,,1002,1118\nP000103_T1_V2.xlsm,Annotations,23,Person B,Place_Down,Alcohol_Bottle,,,1119,1167\nP000103_T1_V2.xlsm,Annotations,24,Person B,Idle,,,,1168,1323\nP000103_T1_V2.xlsm,Annotations,25,Person B,Grasp,Glass,,,1324,1377\nP000103_T1_V2.xlsm,Annotations,26,Person B,Hold,Glass,,,1378,1600\nP000103_T1_V2.xlsm,Annotations,27,Person B,Hold,Glass,Collaborative_Pour,with Person C,1601,1771\nP000103_T1_V2.xlsm,Annotations,28,Person B,Place_Down,Glass,,,1772,1832\nP000103_T1_V2.xlsm,Annotations,29,Person B,Idle,,,,1833,2110\nP000103_T1_V2.xlsm,Annotations,32,Person C,Idle,,,,0,121\nP000103_T1_V2.xlsm,Annotations,33,Person C,Grasp,Shaker,,,122,148\nP000103_T1_V2.xlsm,Annotations,34,Person C,Twist,Shaker_Lid,,,149,228\nP000103_T1_V2.xlsm,Annotations,35,Person C,Place_Down,Shaker_Lid,,,229,261\nP000103_T1_V2.xlsm,Annotations,36,Person C,Hold,Shaker,,,262,299\nP000103_T1_V2.xlsm,Annotations,37,Person C,Hold,Shaker,Collaborative_Drop,with Person B,300,350\nP000103_T1_V2.xlsm,Annotations,38,Person C,Grasp,Sugar,,,351,398\nP000103_T1_V2.xlsm,Annotations,39,Person C,Drop,Sugar,,,399,463\nP000103_T1_V2.xlsm,Annotations,40,Person C,Handover,Muddler,,,464,497\nP000103_T1_V2.xlsm,Annotations,41,Person C,Squash,,,,498,675\nP000103_T1_V2.xlsm,Annotations,42,Person C,Place_Down,Muddler,,,676,698\nP000103_T1_V2.xlsm,Annotations,43,Person C,Grasp,Shaker,,,699,728\nP000103_T1_V2.xlsm,Annotations,44,Person C,Hold,Shaker,Collaborative_Pour,with Person B,729,1023\nP000103_T1_V2.xlsm,Annotations,45,Person C,Place_Down,Shaker,,,1024,1076\nP000103_T1_V2.xlsm,Annotations,46,Person C,Grasp,Shaker_Lid,,,1077,1124\nP000103_T1_V2.xlsm,Annotations,47,Person C,Twist,Shaker_Lid,,,1125,1200\nP000103_T1_V2.xlsm,Annotations,48,Person C,Shake,Shaker,,,1201,1449\nP000103_T1_V2.xlsm,Annotations,49,Person C,Twist,Shaker_Lid,,,1450,1526\nP000103_T1_V2.xlsm,Annotations,50,Person C,Place_Down,Shaker_Lid,,,1527,1579\nP000103_T1_V2.xlsm,Annotations,51,Person C,Grasp,Shaker,,,1580,1600\nP000103_T1_V2.xlsm,Annotations,52,Person C,Hold,Shaker,Collaborative_Pour,with Person B,1601,1782\nP000103_T1_V2.xlsm,Annotations,53,Person C,Hold,Shaker,Collaborative_Pour,with Person B,1783,1952\nP000103_T1_V2.xlsm,Annotations,54,Person C,Place_Down,Shaker,,,1953,2009\nP000103_T1_V2.xlsm,Annotations,55,Person C,Idle,,,,2010,2110\nP000200_T1_V1.xlsm,Annotations,5,Person B,Idle,,,,0,106\nP000200_T1_V1.xlsm,Annotations,6,Person B,Grasp,Knife,,,107,158\nP000200_T1_V1.xlsm,Annotations,7,Person B,Cut,Lime,,,159,246\nP000200_T1_V1.xlsm,Annotations,8,Person B,Place_Down,Knife,,,247,375\nP000200_T1_V1.xlsm,Annotations,9,Person B,Push,Cutting_Board,,,376,434\nP000200_T1_V1.xlsm,Annotations,10,Person B,Grasp,Shaker,,,435,481\nP000200_T1_V1.xlsm,Annotations,11,Person B,Twist,Shaker_Lid,,,482,544\nP000200_T1_V1.xlsm,Annotations,12,Person B,Place_Down,Shaker_Lid,,,545,589\nP000200_T1_V1.xlsm,Annotations,13,Person B,Grasp,Sugar_Container,,,590,609\nP000200_T1_V1.xlsm,Annotations,14,Person B,Place_Down,Sugar_Container,,,610,638\nP000200_T1_V1.xlsm,Annotations,15,Person B,Grasp,Sugar,,,639,672\nP000200_T1_V1.xlsm,Annotations,16,Person B,Drop,Sugar,,,673,701\nP000200_T1_V1.xlsm,Annotations,17,Person B,Grasp,Ice,,,702,781\nP000200_T1_V1.xlsm,Annotations,18,Person B,Drop,Ice,,,782,809\nP000200_T1_V1.xlsm,Annotations,19,Person B,Grasp,Ice,,,810,852\nP000200_T1_V1.xlsm,Annotations,20,Person B,Drop,Ice,,,853,879\nP000200_T1_V1.xlsm,Annotations,21,Person B,Grasp,Lime,,,880,910\nP000200_T1_V1.xlsm,Annotations,22,Person B,Drop,Lime,,,911,941\nP000200_T1_V1.xlsm,Annotations,23,Person B,Place_Down,Shaker,,,942,968\nP000200_T1_V1.xlsm,Annotations,24,Person B,Grasp,Alcohol_Bottle,,,969,1007\nP000200_T1_V1.xlsm,Annotations,25,Person B,Twist,Bottle_Cap,,,1008,1100\nP000200_T1_V1.xlsm,Annotations,26,Person B,Place_Down,Bottle_Cap,,,1101,1129\nP000200_T1_V1.xlsm,Annotations,27,Person B,Pour,Alcohol,,,1130,1285\nP000200_T1_V1.xlsm,Annotations,28,Person B,Grasp,Bottle_Cap,,,1286,1324\nP000200_T1_V1.xlsm,Annotations,29,Person B,Twist,Bottle_Cap,,,1325,1433\nP000200_T1_V1.xlsm,Annotations,30,Person B,Place_Down,Alcohol_Bottle,,,1434,1482\nP000200_T1_V1.xlsm,Annotations,31,Person B,Grasp,Shaker,,,1483,1534\nP000200_T1_V1.xlsm,Annotations,32,Person B,Twist,Shaker_Lid,,,1535,1603\nP000200_T1_V1.xlsm,Annotations,33,Person B,Shake,Shaker,,,1604,1894\nP000200_T1_V1.xlsm,Annotations,34,Person B,Twist,Shaker_Lid,,,1895,1970\nP000200_T1_V1.xlsm,Annotations,35,Person B,Place_Down,Shaker_Lid,,,1971,2010\nP000200_T1_V1.xlsm,Annotations,36,Person B,Place_Down,Shaker,,,2011,2046\nP000200_T1_V1.xlsm,Annotations,37,Person B,Grasp,Glass,,,2047,2070\nP000200_T1_V1.xlsm,Annotations,38,Person B,Place_Down,Glass,,,2071,2094\nP000200_T1_V1.xlsm,Annotations,39,Person B,Grasp,Shaker,,,2095,2132\nP000200_T1_V1.xlsm,Annotations,40,Person B,Pour,Cocktail,,,2133,2370\nP000200_T1_V1.xlsm,Annotations,41,Person B,Place_Down,Shaker,,,2371,2426\nP000200_T1_V1.xlsm,Annotations,42,Person B,Idle,,,,2427,2500\nP000301_T1_V1.xlsm,Annotations,5,Person B,Idle,,,,0,75\nP000301_T1_V1.xlsm,Annotations,6,Person B,Grasp,Shaker,,,76,136\nP000301_T1_V1.xlsm,Annotations,7,Person B,Handover,Shaker,Collaborative_Handover,with Person C,137,185\nP000301_T1_V1.xlsm,Annotations,8,Person B,Twist,Shaker_Lid,Collaborative_Twist,with Person C,186,259\nP000301_T1_V1.xlsm,Annotations,9,Person B,Place_Down,Shaker_Lid,,,260,294\nP000301_T1_V1.xlsm,Annotations,10,Person B,Grasp,Knife,,,295,323\nP000301_T1_V1.xlsm,Annotations,11,Person B,Cut,Lime,,,324,414\nP000301_T1_V1.xlsm,Annotations,12,Person B,Place_Down,Knife,,,415,428\nP000301_T1_V1.xlsm,Annotations,13,Person B,Grasp,Lime,,,429,460\nP000301_T1_V1.xlsm,Annotations,14,Person B,Drop,Lime,Collaborative_Drop,with Person C,461,490\nP000301_T1_V1.xlsm,Annotations,15,Person B,Grasp,Sugar,,,491,554\nP000301_T1_V1.xlsm,Annotations,16,Person B,Drop,Sugar,Collaborative_Drop,with Person C,555,643\nP000301_T1_V1.xlsm,Annotations,17,Person B,Grasp,Muddler,,,644,709\nP000301_T1_V1.xlsm,Annotations,18,Person B,Squash,,,,710,932\nP000301_T1_V1.xlsm,Annotations,19,Person B,Place_Down,Muddler,,,933,1000\nP000301_T1_V1.xlsm,Annotations,20,Person B,Grasp,Alcohol_Bottle,,,1001,1046\nP000301_T1_V1.xlsm,Annotations,21,Person B,Twist,Bottle_Cap,,,1047,1171\nP000301_T1_V1.xlsm,Annotations,22,Person B,Pour,Alcohol,Collaborative_Pour,with Person C,1172,1357\nP000301_T1_V1.xlsm,Annotations,23,Person B,Hold,Alcohol_Bottle,,,1358,1454\nP000301_T1_V1.xlsm,Annotations,24,Person B,Twist,Bottle_Cap,,,1455,1642\nP000301_T1_V1.xlsm,Annotations,25,Person B,Place_Down,Alcohol_Bottle,,,1643,1687\nP000301_T1_V1.xlsm,Annotations,26,Person B,Idle,,,,1688,1849\nP000301_T1_V1.xlsm,Annotations,27,Person B,Twist,Shaker_Lid,,,1850,1908\nP000301_T1_V1.xlsm,Annotations,28,Person B,Place_Down,Shaker,,,1909,1934\nP000301_T1_V1.xlsm,Annotations,29,Person B,Idle,,,,1935,2042\nP000301_T1_V1.xlsm,Annotations,30,Person B,Handover,Shaker,Collaborative_Handover,with Person C,2043,2069\nP000301_T1_V1.xlsm,Annotations,31,Person B,Pour,Cocktail,,,2070,2177\nP000301_T1_V1.xlsm,Annotations,32,Person B,Place_Down,Shaker,,,2178,2215\nP000301_T1_V1.xlsm,Annotations,33,Person B,Idle,,,,2216,2300\nP000301_T1_V1.xlsm,Annotations,36,Person C,Idle,,,,0,130\nP000301_T1_V1.xlsm,Annotations,37,Person C,Handover,Shaker,Collaborative_Handover,with Person B,131,193\nP000301_T1_V1.xlsm,Annotations,38,Person C,Hold,Shaker,Collaborative_Twist,with Person B,194,252\nP000301_T1_V1.xlsm,Annotations,39,Person C,Hold,Shaker,,,253,459\nP000301_T1_V1.xlsm,Annotations,40,Person C,Hold,Shaker,Collaborative_Drop,with Person B,460,487\nP000301_T1_V1.xlsm,Annotations,41,Person C,Hold,Shaker,,,488,550\nP000301_T1_V1.xlsm,Annotations,42,Person C,Hold,Shaker,Collaborative_Drop,,551,583\nP000301_T1_V1.xlsm,Annotations,43,Person C,Place_Down,Shaker,,,584,638\nP000301_T1_V1.xlsm,Annotations,44,Person C,Idle,,,,639,1058\nP000301_T1_V1.xlsm,Annotations,45,Person C,Grasp,Shaker,,,1059,1123\nP000301_T1_V1.xlsm,Annotations,46,Person C,Hold,Shaker,Collaborative_Pour,with Person B,1124,1346\nP000301_T1_V1.xlsm,Annotations,47,Person C,Hold,Shaker,,,1347,1393\nP000301_T1_V1.xlsm,Annotations,48,Person C,Place_Down,Shaker,,,1394,1425\nP000301_T1_V1.xlsm,Annotations,49,Person C,Grasp,Shaker_Lid,,,1426,1444\nP000301_T1_V1.xlsm,Annotations,50,Person C,Twist,Shaker_Lid,,,1445,1600\nP000301_T1_V1.xlsm,Annotations,51,Person C,Shake,Shaker,,,1601,1853\nP000301_T1_V1.xlsm,Annotations,52,Person C,Hold,Shaker,Collaborative_Twist,with Person B,1854,1907\nP000301_T1_V1.xlsm,Annotations,53,Person C,Pour,Cocktail,,,1908,2029\nP000301_T1_V1.xlsm,Annotations,54,Person C,Handover,Shaker,Collaborative_Handover,with Person B,2030,2081\nP000301_T1_V1.xlsm,Annotations,55,Person C,Idle,,,,2082,2300\nP010203_T1_V1.xlsm,Annotations,3,Person A,Idle,,,,0,153\nP010203_T1_V1.xlsm,Annotations,4,Person A,Grasp,Shaker,,,154,195\nP010203_T1_V1.xlsm,Annotations,5,Person A,Twist,Shaker_Lid,,,196,236\nP010203_T1_V1.xlsm,Annotations,6,Person A,Place_Down,Shaker_Lid,,,237,282\nP010203_T1_V1.xlsm,Annotations,7,Person A,Hold,Shaker,,,283,323\nP010203_T1_V1.xlsm,Annotations,8,Person A,Grasp,Sugar,,,324,357\nP010203_T1_V1.xlsm,Annotations,9,Person A,Drop,Sugar,,,358,400\nP010203_T1_V1.xlsm,Annotations,10,Person A,Hold,Shaker,,,401,428\nP010203_T1_V1.xlsm,Annotations,11,Person A,Hold,Shaker,Collaborative_Drop,with Person C,429,489\nP010203_T1_V1.xlsm,Annotations,12,Person A,Place_Down,Shaker,,,490,524\nP010203_T1_V1.xlsm,Annotations,13,Person A,Handover,Muddler,Collaborative_Handover,with Person B,525,570\nP010203_T1_V1.xlsm,Annotations,14,Person A,Grasp,Shaker,,,571,598\nP010203_T1_V1.xlsm,Annotations,15,Person A,Squash,,,,599,882\nP010203_T1_V1.xlsm,Annotations,16,Person A,Place_Down,Muddler,,,883,918\nP010203_T1_V1.xlsm,Annotations,17,Person A,Hold,Shaker,Collaborative_Pour,with Person B,919,1088\nP010203_T1_V1.xlsm,Annotations,18,Person A,Grasp,Shaker_Lid,,,1089,1130\nP010203_T1_V1.xlsm,Annotations,19,Person A,Twist,Shaker_Lid,,,1131,1189\nP010203_T1_V1.xlsm,Annotations,20,Person A,Shake,Shaker,,,1190,1353\nP010203_T1_V1.xlsm,Annotations,21,Person A,Hold,Shaker,,,1354,1386\nP010203_T1_V1.xlsm,Annotations,22,Person A,Twist,Shaker_Lid,,,1387,1434\nP010203_T1_V1.xlsm,Annotations,23,Person A,Place_Down,Shaker_Lid,,,1435,1482\nP010203_T1_V1.xlsm,Annotations,24,Person A,Handover,Shaker,Collaborative_Handover,with Person B,1483,1528\nP010203_T1_V1.xlsm,Annotations,25,Person A,Idle,,,,1529,1566\nP010203_T1_V1.xlsm,Annotations,26,Person A,Grasp,Glass,,,1567,1610\nP010203_T1_V1.xlsm,Annotations,27,Person A,Hold,Glass,,,1611,1673\nP010203_T1_V1.xlsm,Annotations,28,Person A,Hold,Glass,Collaborative_Pour,with Person B,1674,1826\nP010203_T1_V1.xlsm,Annotations,29,Person A,Hold,Glass,,,1827,2070\nP010203_T1_V1.xlsm,Annotations,32,Person B,Idle,,,,0,197\nP010203_T1_V1.xlsm,Annotations,33,Person B,Grasp,Sugar_Container,,,198,233\nP010203_T1_V1.xlsm,Annotations,34,Person B,Place_Down,Sugar_Container,,,234,266\nP010203_T1_V1.xlsm,Annotations,35,Person B,Idle,,,,267,478\nP010203_T1_V1.xlsm,Annotations,36,Person B,Grasp,Muddler,,,479,524\nP010203_T1_V1.xlsm,Annotations,37,Person B,Handover,Muddler,Collaborative_Handover,with Person A,525,564\nP010203_T1_V1.xlsm,Annotations,38,Person B,Idle,,,,565,614\nP010203_T1_V1.xlsm,Annotations,39,Person B,Grasp,Alcohol_Bottle,,,615,650\nP010203_T1_V1.xlsm,Annotations,40,Person B,Hold,Alcohol_Bottle,Collaborative_Twist,with Person C,651,734\nP010203_T1_V1.xlsm,Annotations,41,Person B,Hold,Alcohol_Bottle,,,735,915\nP010203_T1_V1.xlsm,Annotations,42,Person B,Pour,Alcohol,Collaborative_Pour,with Person A,916,1075\nP010203_T1_V1.xlsm,Annotations,43,Person B,Hold,Alcohol_Bottle,Collaborative_Twist,with Person C,1076,1204\nP010203_T1_V1.xlsm,Annotations,44,Person B,Place_Down,Alcohol_Bottle,,,1205,1266\nP010203_T1_V1.xlsm,Annotations,45,Person B,Idle,,,,1267,1481\nP010203_T1_V1.xlsm,Annotations,46,Person B,Handover,Shaker,Collaborative_Handover,with Person A,1482,1531\nP010203_T1_V1.xlsm,Annotations,47,Person B,Pour,Cocktail,,,1532,1665\nP010203_T1_V1.xlsm,Annotations,48,Person B,Pour,Cocktail,Collaborative_Pour,with Person A,1666,1820\nP010203_T1_V1.xlsm,Annotations,49,Person B,Push,Glass,,,1821,1850\nP010203_T1_V1.xlsm,Annotations,50,Person B,Pour,Cocktail,,,1851,1940\nP010203_T1_V1.xlsm,Annotations,51,Person B,Place_Down,Shaker,,,1941,2004\nP010203_T1_V1.xlsm,Annotations,52,Person B,Idle,,,,2005,2070\nP010203_T1_V1.xlsm,Annotations,55,Person C,Idle,,,,0,148\nP010203_T1_V1.xlsm,Annotations,56,Person C,Grasp,Knife,,,149,200\nP010203_T1_V1.xlsm,Annotations,57,Person C,Cut,Lime,,,201,275\nP010203_T1_V1.xlsm,Annotations,58,Person C,Place_Down,Knife,,,276,306\nP010203_T1_V1.xlsm,Annotations,59,Person C,Idle,,,,307,399\nP010203_T1_V1.xlsm,Annotations,60,Person C,Grasp,Lime,,,400,436\nP010203_T1_V1.xlsm,Annotations,61,Person C,Drop,Lime,Collaborative_Drop,with Person A,437,486\nP010203_T1_V1.xlsm,Annotations,62,Person C,Idle,,,,487,633\nP010203_T1_V1.xlsm,Annotations,63,Person C,Twist,Bottle_Cap,Collaborative_Twist,with Person B,634,733\nP010203_T1_V1.xlsm,Annotations,64,Person C,Place_Down,Bottle_Cap,,,734,766\nP010203_T1_V1.xlsm,Annotations,65,Person C,Idle,,,,767,1040\nP010203_T1_V1.xlsm,Annotations,66,Person C,Grasp,Bottle_Cap,,,1041,1091\nP010203_T1_V1.xlsm,Annotations,67,Person C,Twist,Bottle_Cap,Collaborative_Twist,with Person B,1092,1208\nP010203_T1_V1.xlsm,Annotations,68,Person C,Idle,,,,1209,2070\n'
if GT_CSV is None:
    GT_CSV = WORK_DIR / 'ground_truth_actions.csv'
    GT_CSV.write_text(EMBEDDED_GT_CSV, encoding='utf-8')
if not GT_CSV.is_file():
    raise FileNotFoundError(GT_CSV)
print(f'Ground truth: {GT_CSV}')


## Valutatore
La cella seguente definisce il valutatore standard Python. Copia corrispondente a `evaluation/Ground/evaluate_actions.py`.
Per finestre sovrapposte si sceglie il centro più vicino; a parità vince la precedente. L’abbinamento persone resta fisso nella clip e identico per tutte le medie.


In [ ]:
"""Frame-level Ground evaluation with explicitly optimistic action-based identity matching.

No external dependencies. Input ZIP is read without extracting or executing content.
"""
import argparse
import csv
import itertools
import json
import re
import zipfile
from collections import Counter, defaultdict
from pathlib import Path

ACTIONS = ['cut', 'drop', 'grasp', 'handover', 'hold', 'idle', 'place down',
           'pour', 'push', 'shake', 'squash', 'twist']
NAME = re.compile(r'(P\d+_T\d+_V\d+)_C(\d+)_scene(\d+)_part(\d+)_(person_\d+)_actions.json')
WINDOW = re.compile(r'frames_(\d+)_to_(\d+)')
SUBSETS = [('', set()), ('_NO_DROP', {'drop'}), ('_NO_IDLE', {'idle'}),
           ('_NO_IDLE_DROP', {'idle', 'drop'}),
           ('_NO_IDLE_HOLD', {'idle', 'hold'}),
           ('_NO_IDLE_HOLD_DROP', {'idle', 'hold', 'drop'})]


def write_csv(path, rows, fields=None):
    with path.open('w', newline='') as stream:
        writer = csv.DictWriter(stream, fieldnames=fields or list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)


def class_metrics(matrix):
    rows = []
    for label in ACTIONS:
        tp = matrix[label, label]
        support = sum(n for (a, b), n in matrix.items() if a == label)
        predicted = sum(n for (a, b), n in matrix.items() if b == label)
        covered = support - matrix[label, 'no_prediction']
        precision = tp / predicted if predicted else 0
        recall = tp / support if support else 0
        rows.append(dict(action=label, support_frames=support, predicted_frames=predicted,
                         tp=tp, fp=predicted - tp, fn=support - tp,
                         covered_frames=covered, coverage_ratio=covered / support if support else 0,
                         precision=precision, recall=recall,
                         f1=2 * tp / (predicted + support) if predicted + support else 0))
    return rows


def averages(rows):
    result = []
    for suffix, excluded in SUBSETS:
        selected = [r for r in rows if r['action'] not in excluded]
        support = sum(r['support_frames'] for r in selected)
        tp, fp, fn = (sum(r[k] for r in selected) for k in ('tp', 'fp', 'fn'))
        coverage = sum(r['covered_frames'] for r in selected) / support if support else 0
        for mode in ('MACRO', 'MICRO', 'WEIGHTED'):
            if mode == 'MICRO':
                values = dict(precision=tp / (tp + fp) if tp + fp else 0,
                              recall=tp / (tp + fn) if tp + fn else 0,
                              f1=2 * tp / (2 * tp + fp + fn) if 2 * tp + fp + fn else 0)
            else:
                weights = [1 if mode == 'MACRO' else r['support_frames'] for r in selected]
                denominator = sum(weights)
                values = {k: sum(r[k] * w for r, w in zip(selected, weights)) / denominator
                          if denominator else 0 for k in ('precision', 'recall', 'f1')}
            result.append(dict(average=mode + '_AVG' + suffix, support_frames=support,
                               tp=tp if mode == 'MICRO' else '',
                               fp=fp if mode == 'MICRO' else '',
                               fn=fn if mode == 'MICRO' else '', coverage_ratio=coverage, **values))
    return result


def collapse_trace(labels, excluded):
    """Filter the selected vocabulary, then merge equal consecutive labels."""
    return [label for label, _ in itertools.groupby(x for x in labels if x not in excluded)]


def sequence_scores(reference, prediction):
    """Unit-cost Levenshtein and order-preserving longest common subsequence."""
    edit = list(range(len(prediction) + 1))
    lcs = [0] * (len(prediction) + 1)
    for i, expected in enumerate(reference, 1):
        next_edit, next_lcs = [i], [0]
        for j, actual in enumerate(prediction, 1):
            next_edit.append(min(edit[j] + 1, next_edit[-1] + 1,
                                 edit[j - 1] + (expected != actual)))
            next_lcs.append(lcs[j - 1] + 1 if expected == actual else max(lcs[j], next_lcs[-1]))
        edit, lcs = next_edit, next_lcs
    distance, matched = edit[-1], lcs[-1]
    longest = max(len(reference), len(prediction))
    return dict(gt_events=len(reference), predicted_events=len(prediction),
                exact_match=int(reference == prediction), edit_distance=distance,
                edit_similarity=1 - distance / longest if longest else 1,
                lcs_matches=matched,
                lcs_precision=matched / len(prediction) if prediction else int(not reference),
                lcs_recall=matched / len(reference) if reference else int(not prediction),
                lcs_f1=2 * matched / (len(reference) + len(prediction)) if longest else 1)


def trace_evaluation(gt, predictions):
    traces, averages = [], []
    for suffix, excluded in SUBSETS:
        subset = 'ALL' + suffix
        for video, people in sorted(gt.items()):
            for person, frames in sorted(people.items()):
                ordered = sorted(frames)
                if ordered != list(range(ordered[0], ordered[-1] + 1)):
                    raise ValueError(f'Trace has GT gaps: {video}/{person}; explicit gap alignment required')
                reference = collapse_trace((frames[f] for f in ordered), excluded)
                prediction = collapse_trace((predictions[video, person].get(f, 'no_prediction')
                                              for f in ordered), excluded)
                traces.append(dict(subset=subset, video=video, person=person,
                                   **sequence_scores(reference, prediction),
                                   gt_sequence=json.dumps(reference), predicted_sequence=json.dumps(prediction)))
        rows = [r for r in traces if r['subset'] == subset]
        averages.append(dict(subset=subset, traces=len(rows),
                             exact_trace_accuracy=sum(r['exact_match'] for r in rows) / len(rows),
                             mean_edit_distance=sum(r['edit_distance'] for r in rows) / len(rows),
                             mean_edit_similarity=sum(r['edit_similarity'] for r in rows) / len(rows),
                             mean_lcs_precision=sum(r['lcs_precision'] for r in rows) / len(rows),
                             mean_lcs_recall=sum(r['lcs_recall'] for r in rows) / len(rows),
                             mean_lcs_f1=sum(r['lcs_f1'] for r in rows) / len(rows)))
    return traces, averages


def main(argv=None):
    parser = argparse.ArgumentParser(description=__doc__)
    parser.add_argument('predictions', type=Path)
    parser.add_argument('--ground-truth', type=Path,
                        default=Path(globals().get('__file__', 'evaluate_actions.py')).with_name('ground_truth_actions.csv'))
    parser.add_argument('--output', type=Path, required=True)
    args = parser.parse_args(argv)
    args.output.mkdir(parents=True, exist_ok=True)
    gt = defaultdict(lambda: defaultdict(dict))
    with args.ground_truth.open() as stream:
        for row in csv.DictReader(stream):
            label = row['verb'].lower().replace('_', ' ')
            if label not in ACTIONS:
                raise ValueError(f'Unsupported GT verb: {label}')
            frames = gt[Path(row['source_file']).stem][row['person']]
            for frame in range(int(row['start']), int(row['end']) + 1):
                if frame in frames:
                    raise ValueError('Overlapping GT intervals')
                frames[frame] = label

    clips = defaultdict(dict)
    with zipfile.ZipFile(args.predictions) as archive:
        for entry in archive.infolist():
            if entry.is_dir() or '__MACOSX' in entry.filename.split('/'):
                continue
            name = Path(entry.filename).name
            if not name.endswith('.json'):
                continue
            match = NAME.fullmatch(name)
            if not match:
                raise ValueError(f'Unrecognized filename: {name}')
            video, camera, scene, part, track = match.groups()
            if camera != '4' or scene != '1' or video not in gt:
                raise ValueError(f'Cannot align video automatically: {name}')
            key = (video, int(part))
            if track in clips[key]:
                raise ValueError(f'Duplicate prediction file: {name}')
            windows = []
            for key_window, value in json.loads(archive.read(entry)).items():
                window = WINDOW.fullmatch(key_window)
                if not window:
                    raise ValueError(f'Invalid window: {key_window}')
                start, end = map(int, window.groups())
                labels = value['actions']
                if not 0 <= start < end <= 300 or len(labels) != 1:
                    raise ValueError(f'Invalid prediction: {name}/{key_window}')
                if labels[0] not in ACTIONS + ['unknown']:
                    raise ValueError(f'Invalid label: {labels[0]}')
                windows.append((start, end, labels[0]))
            # Overlap resolution: nearest window center, earlier start wins ties.
            frames = {}
            for frame in range(300):
                candidates = [w for w in windows if w[0] <= frame < w[1]]
                if candidates:
                    best = min(candidates, key=lambda w: (abs((w[0] + w[1] - 1) / 2 - frame), w[0]))
                    frames[frame] = best[2]
            clips[key][track] = frames

    matrix = Counter()
    predicted_traces = defaultdict(dict)
    mappings, alternatives, video_rows = [], [], []
    for video, people in sorted(gt.items()):
        video_matrix = Counter()
        final = max(max(frames) for frames in people.values())
        for part in range(1, final // 300 + 2):
            offset = (part - 1) * 300
            active = sorted(p for p, frames in people.items() if any(offset <= f < offset + 300 for f in frames))
            tracks = clips.get((video, part), {})
            track_ids = sorted(tracks)
            # Assign injectively; missing tracks are explicit, surplus tracks reported.
            candidates = track_ids + [f'missing_{i}' for i in range(max(0, len(active) - len(track_ids)))]
            scores = []
            for assignment in itertools.permutations(candidates, len(active)):
                score = sum(
                    tracks.get(t, {}).get(f - offset, 'no_prediction') == label
                    for p, t in zip(active, assignment)
                    for f, label in people[p].items() if offset <= f < offset + 300)
                scores.append((score, assignment))
            scores.sort(key=lambda item: (-item[0], item[1]))
            best_score, best = scores[0]
            support = sum(offset <= f < offset + 300 for p in active for f in people[p])
            gap = best_score - scores[1][0] if len(scores) > 1 else None
            for rank, (score, assignment) in enumerate(scores, 1):
                alternatives.append(dict(video=video, part=part, rank=rank,
                                         mapping=json.dumps(dict(zip(active, assignment))),
                                         correct_frames=score, support_frames=support))
            for person, track in zip(active, best):
                mappings.append(dict(video=video, part=part, gt_person=person,
                                     track=track, best_correct_frames=best_score,
                                     next_best_gap_frames=gap, tied=sum(s[0] == best_score for s in scores),
                                     unused_tracks=';'.join(t for t in track_ids if t not in best)))
                for frame, label in people[person].items():
                    if offset <= frame < offset + 300:
                        pred = tracks.get(track, {}).get(frame - offset, 'no_prediction')
                        predicted_traces[video, person][frame] = pred
                        video_matrix[label, pred] += 1
        matrix.update(video_matrix)
        support = sum(video_matrix.values())
        missing = sum(n for (a, b), n in video_matrix.items() if b == 'no_prediction')
        video_classes = class_metrics(video_matrix)
        video_rows.append(dict(video=video, support_frames=support,
                               accuracy=sum(n for (a, b), n in video_matrix.items() if a == b) / support,
                               coverage_ratio=1 - missing / support,
                               macro_f1=sum(r['f1'] for r in video_classes) / len(ACTIONS),
                               no_prediction_frames=missing))

    per_class = class_metrics(matrix)
    aggregate_rows = averages(per_class)
    trace_rows, trace_averages = trace_evaluation(gt, predicted_traces)
    total = sum(matrix.values())
    summary = dict(protocol='optimistic action-based matching per clip; identities not independently verified',
                   alignment='scene1 starts at 0; part offsets 300 frames; GT End inclusive; predictions End exclusive',
                   accuracy=sum(n for (a, b), n in matrix.items() if a == b) / total,
                   macro_f1=sum(r['f1'] for r in per_class) / len(per_class),
                   weighted_f1=sum(r['f1'] * r['support_frames'] for r in per_class) / total,
                   support_person_frames=total,
                   unknown_rate=sum(n for (a, b), n in matrix.items() if b == 'unknown') / total,
                   no_prediction_rate=sum(n for (a, b), n in matrix.items() if b == 'no_prediction') / total,
                   prediction_files=sum(len(v) for v in clips.values()))
    summary.update(videos_evaluated=len(video_rows),
                   accuracy_macro_per_video=sum(r['accuracy'] for r in video_rows) / len(video_rows),
                   coverage_macro_per_video=sum(r['coverage_ratio'] for r in video_rows) / len(video_rows),
                   averages=aggregate_rows, trace_averages=trace_averages)
    write_csv(args.output / 'person_mapping.csv', mappings)
    write_csv(args.output / 'mapping_alternatives.csv', alternatives)
    write_csv(args.output / 'per_class.csv', per_class)
    write_csv(args.output / 'averages.csv', aggregate_rows)
    write_csv(args.output / 'per_video.csv', video_rows)
    write_csv(args.output / 'per_trace.csv', trace_rows)
    write_csv(args.output / 'trace_averages.csv', trace_averages)
    write_csv(args.output / 'confusion_matrix.csv', [dict(gt_action=a, **{b: matrix[a, b] for b in ACTIONS + ['unknown', 'no_prediction']}) for a in ACTIONS])
    (args.output / 'summary.json').write_text(json.dumps(summary, indent=2) + '\n')
    lines = [
        'GROUND ACTION RECOGNITION — FRAME LEVEL',
        'OPTIMISTIC: person identities inferred from agreement with GT actions.',
        'Alignment assumed: 30 FPS, scene1 offset 0, 300 frames per part, GT End inclusive.',
        f'Videos evaluated: {len(video_rows)} | Prediction files: {summary["prediction_files"]}',
        f'Annotated person-frames: {total}',
        f'Accuracy pooled: {summary["accuracy"]:.4f}',
        f'Accuracy macro per video: {summary["accuracy_macro_per_video"]:.4f}',
        f'Coverage macro per video: {summary["coverage_macro_per_video"]:.4f}',
        f'Unknown rate: {summary["unknown_rate"]:.4f}',
        'Subset accuracy / sample Jaccard: same as accuracy for singleton labels; abstentions treated as empty sets.',
        'No L0/L1 hierarchy is defined for Ground.', '',
        f'{"action":<24} {"precision":>9} {"recall":>8} {"f1":>8} {"support":>8} {"tp":>7} {"fp":>7} {"fn":>7} {"coverage":>9}',
    ]
    for row in per_class + [dict(r, action=r['average']) for r in aggregate_rows]:
        lines.append(f'{row["action"]:<24} {row["precision"]:9.4f} {row["recall"]:8.4f} '
                     f'{row["f1"]:8.4f} {row["support_frames"]:8d} {str(row["tp"]):>7} '
                     f'{str(row["fp"]):>7} {str(row["fn"]):>7} {row["coverage_ratio"]:9.4f}')
    lines.extend(['', 'Excluded-class averages retain errors across excluded/included classes as FP/FN.',
                  'Unknown and no_prediction cause FN, not FP for real action classes; unknown counts as covered.',
                  'Macro averages give equal weight to classes (including zero scores); weighted averages use GT support.',
                  'No GT videos are skipped; missing predictions are scored as no_prediction.',
                  'These are frame classification scores, not temporal event detection metrics.'])
    lines.extend(['', 'FULL PERSON TRACES — ORDER ONLY, CONSECUTIVE DUPLICATES COLLAPSED',
                  'One trace per GT person per original video; clips joined using the same inferred mapping.',
                  'No total ordering is imposed between simultaneous actions of different people.',
                  'Unknown/no_prediction retained as error events; no smoothing or minimum duration filter.',
                  'Sequence subsets filter labels on BOTH sides before collapsing: a different protocol from frame averages.',
                  'Levenshtein uses unit insertion/deletion/substitution costs; similarity = 1 - distance / max(lengths).',
                  'LCS matches actions in order, without requiring adjacency. Averages give equal weight to traces.',
                  'Empty/empty sequences score 1; empty/nonempty scores 0 for normalized similarities.',
                  f'{"subset":<24} {"traces":>6} {"exact":>8} {"edit_sim":>9} {"LCS_P":>8} {"LCS_R":>8} {"LCS_F1":>8}'])
    for row in trace_averages:
        lines.append(f'{row["subset"]:<24} {row["traces"]:6d} {row["exact_trace_accuracy"]:8.4f} '
                     f'{row["mean_edit_similarity"]:9.4f} {row["mean_lcs_precision"]:8.4f} '
                     f'{row["mean_lcs_recall"]:8.4f} {row["mean_lcs_f1"]:8.4f}')
    (args.output / 'evaluation_results.txt').write_text('\n'.join(lines) + '\n')
    print(json.dumps(summary, indent=2))



## Confronto con e senza Drop
L’eventuale miglioramento è un effetto del sottoinsieme di classi riportato, non un miglioramento del modello. Conservare sempre anche i risultati completi.


## Valutazione della trace completa
Una trace è la sequenza di una persona GT nell’intero video originale. Le clip vengono riunite usando il mapping già inferito, senza riottimizzarlo per le sequenze. Non imponiamo un ordine totale alle azioni simultanee di persone diverse.

Uniamo le ripetizioni consecutive: `hold, hold, pour, pour, hold` diventa `hold → pour → hold`. Non eliminiamo eventi brevi, `unknown` o lacune `no_prediction`.

- **Exact trace accuracy:** quota di trace identiche al GT.
- **Edit similarity:** `1 − Levenshtein / max(lunghezza GT, lunghezza predetta)`. Conta inserimenti, cancellazioni e sostituzioni, costo 1; da 0 a 1.
- **LCS precision/recall/F1:** azioni abbinate in ordine tramite la più lunga sottosequenza comune. Penalizza eventi extra o mancanti senza richiedere adiacenza.

Le medie assegnano uguale peso a ogni trace. I tempi e le durate sono valutati separatamente dalle metriche per frame. Le varianti senza Drop/Idle/Hold filtrano quelle etichette da **entrambe** le sequenze e poi comprimono: è diverso dall’esclusione dalle sole medie per frame. Due sequenze vuote valgono 1; una sola vuota vale 0.

I CSV `per_trace.csv` e `trace_averages.csv` includono tutte le sequenze e i risultati. Il mapping resta ottimistico e può cambiare ai confini delle clip; l’identità della trace ricostruita non è verificata indipendentemente.


## Object-centric event log

L’esecuzione finale usa **le predizioni e le associazioni persona–oggetto**, senza usare il mapping GT. Richiede la cartella `person_object_associations` prodotta dalla pipeline oppure uno ZIP della stessa cartella. Il solo ZIP delle azioni non contiene gli oggetti.

Ogni riga è un intervallo in cui restano uguali azione e insieme degli oggetti associati. Timestamp di inizio e fine relativi al video originale (`HH:MM:SS.mmm`), fine esclusa; presenti anche secondi numerici e durata. Un evento si spezza quando cambia un oggetto associato, anche se il verbo resta uguale: non è quindi lo stesso livello di segmentazione della trace di soli verbi.

Una colonna `object::<tipo>` per ciascuna classe osservata contiene una lista JSON di ID; per esempio `object::glass` può contenere due bicchieri. Celle vuote = `[]`. `object::person` identifica l'attore. Le etichette dei tipi sono quelle della detection, senza introdurre una tassonomia nuova.

**Gli ID sono locali alla clip**, nella forma `clip::track`: non assumiamo identità fisica fra clip né fra video. I timestamp relativi di video diversi non sono confrontabili come orari reali. Le associazioni geometriche indicano oggetti associati, non provano la manipolazione. Si mantengono soltanto le associazioni di frame supportate da una finestra accettata dello stadio di association.

L'esportazione produce un CSV ampio, una tabella oggetti, una tabella relazioni evento–oggetto e metadati. È una rappresentazione object-centric tabellare, non un file OCEL 2.0. `unknown` e `no_prediction` restano espliciti; nessun oggetto viene inventato quando l'associazione è vuota. File associazioni mancanti bloccano l'esportazione per evitare un log incompleto.


In [ ]:
"""Export a wide object-centric CSV with clip-scoped object identities.

Relative timestamps; association evidence is not proof of manipulation.
"""
import csv
import json
import re
import zipfile
from collections import defaultdict
from pathlib import Path


def relative_timestamp(frame, fps=30):
    milliseconds = round(frame * 1000 / fps)
    seconds, milliseconds = divmod(milliseconds, 1000)
    minutes, seconds = divmod(seconds, 60)
    hours, minutes = divmod(minutes, 60)
    return f'{hours:02d}:{minutes:02d}:{seconds:02d}.{milliseconds:03d}'


def export_object_event_log(predictions_zip, associations_path, output_dir, fps=30):
    """Read model predictions plus association JSONs; never use GT person matching."""
    associations_path, output_dir = Path(associations_path), Path(output_dir)
    if fps != 30:
        raise ValueError('This Ground exporter assumes 300-frame parts at 30 FPS.')
    association_data = {}

    def add_association(name, data):
        if name in association_data:
            raise ValueError(f'Duplicate association file: {name}')
        association_data[name] = data

    if associations_path.is_dir():
        for path in associations_path.rglob('*_associations.json'):
            add_association(path.name, json.loads(path.read_text()))
    elif associations_path.is_file() and zipfile.is_zipfile(associations_path):
        with zipfile.ZipFile(associations_path) as archive:
            for entry in archive.infolist():
                if '__MACOSX' not in entry.filename.split('/') and entry.filename.endswith('_associations.json'):
                    add_association(Path(entry.filename).name, json.loads(archive.read(entry)))
    else:
        raise FileNotFoundError(f'Association folder or ZIP not found: {associations_path}')

    events, objects, object_types = [], {}, set()
    pattern = re.compile(r'((P\d+_T\d+_V\d+)_C4_scene1_part(\d+))_(person_\d+)_actions.json')
    with zipfile.ZipFile(predictions_zip) as archive:
        inputs = []
        for entry in archive.infolist():
            if entry.is_dir() or '__MACOSX' in entry.filename.split('/') or not entry.filename.endswith('.json'):
                continue
            match = pattern.fullmatch(Path(entry.filename).name)
            if not match:
                raise ValueError(f'Unsupported prediction filename: {entry.filename}')
            clip, video, part, person = match.groups()
            association_name = clip + '_associations.json'
            if association_name not in association_data:
                raise FileNotFoundError(f'Missing association file: {association_name}; no partial log exported')
            inputs.append((clip, video, int(part), person, json.loads(archive.read(entry))))
    seen = set()
    for clip, video, part, person, predictions in sorted(inputs):
        if (clip, person) in seen:
            raise ValueError(f'Duplicate clip/person: {clip}/{person}')
        seen.add((clip, person))
        data = association_data[clip + '_associations.json']
        if 'frame_associations' not in data or 'window_associations' not in data:
            raise ValueError(f'Incomplete association schema: {clip}')
        accepted = defaultdict(list)
        for item in data['window_associations']:
            if item.get('accepted') and item['person_id'] == person:
                accepted[item['object_id']].append((item['start_frame'], item['end_frame']))
        windows = []
        for key, value in predictions.items():
            match = re.fullmatch(r'frames_(\d+)_to_(\d+)', key)
            if not match or len(value.get('actions', [])) != 1:
                raise ValueError(f'Invalid prediction: {clip}/{person}/{key}')
            start, end = map(int, match.groups())
            if not 0 <= start < end <= 300:
                raise ValueError(f'Invalid interval: {key}')
            windows.append((start, end, value['actions'][0]))
        if not windows:
            continue
        person_oid = f'{clip}::{person}'
        objects[person_oid] = dict(object_id=person_oid, object_type='person', clip_id=clip, local_track_id=person)
        states = []
        # Missing leading/intermediate predictions are explicit; trailing extent is unknown.
        for frame in range(max(w[1] for w in windows)):
            candidates = [w for w in windows if w[0] <= frame < w[1]]
            action = min(candidates, key=lambda w: (abs((w[0] + w[1] - 1) / 2 - frame), w[0]))[2] if candidates else 'no_prediction'
            links = set()
            for item in data['frame_associations'].get(str(frame), []):
                if item['person_id'] != person:
                    continue
                local_id = item['object_id']
                if not any(start <= frame < end for start, end in accepted.get(local_id, [])):
                    continue
                kind = item.get('object_class') or 'unknown'
                if kind == 'person':
                    raise ValueError('Non-person association expected')
                oid = f'{clip}::{local_id}'
                obj = dict(object_id=oid, object_type=kind, clip_id=clip, local_track_id=local_id)
                if oid in objects and objects[oid] != obj:
                    raise ValueError(f'Inconsistent object type: {oid}')
                objects[oid] = obj
                object_types.add(kind)
                links.add((kind, oid))
            states.append((action, tuple(sorted(links))))
        start = 0
        for end in range(1, len(states) + 1):
            if end < len(states) and states[end] == states[start]:
                continue
            action, links = states[start]
            offset = (part - 1) * 300
            row = dict(video_id=video, clip_id=clip, person_id=person,
                       timestamp=relative_timestamp(offset + start, fps),
                       end_timestamp=relative_timestamp(offset + end, fps),
                       start_seconds=(offset + start) / fps, end_seconds=(offset + end) / fps,
                       duration_seconds=(end - start) / fps, action=action,
                       local_start_frame=start, local_end_frame=end,
                       association_status='associated' if links else 'no_accepted_association',
                       **{'object::person': json.dumps([person_oid])})
            for kind, oid in links:
                row.setdefault('object::' + kind, []).append(oid)
            for key, value in list(row.items()):
                if key.startswith('object::') and isinstance(value, list):
                    row[key] = json.dumps(value)
            events.append(row)
            start = end
    if not events:
        raise ValueError('No events to export')
    events.sort(key=lambda r: (r['video_id'], r['start_seconds'], r['clip_id'], r['person_id']))
    relations = []
    for index, row in enumerate(events, 1):
        row['event_id'] = f'event_{index:07d}'
        for key, value in row.items():
            if key.startswith('object::'):
                for oid in json.loads(value):
                    relations.append(dict(event_id=row['event_id'], object_id=oid,
                                          qualifier='actor' if key == 'object::person' else 'associated'))
    output_dir.mkdir(parents=True, exist_ok=True)
    base = ['event_id'] + [k for k in events[0] if k != 'event_id' and not k.startswith('object::')]
    fields = base + ['object::person'] + ['object::' + kind for kind in sorted(object_types)]
    for filename, rows, columns in [
        ('object_centric_event_log.csv', events, fields),
        ('event_log_objects.csv', sorted(objects.values(), key=lambda x: x['object_id']), list(next(iter(objects.values())))),
        ('event_object_relations.csv', relations, ['event_id', 'object_id', 'qualifier']),
    ]:
        with (output_dir / filename).open('w', newline='') as stream:
            writer = csv.DictWriter(stream, fieldnames=columns, restval='[]')
            writer.writeheader()
            writer.writerows(rows)
    metadata = dict(events=len(events), objects=len(objects), clip_persons=len(seen),
                    timestamp_basis='relative to original video; 30 FPS; scene1 offset 0; 300-frame parts',
                    segmentation='split on action OR accepted associated object-set change; no smoothing',
                    identity_scope='clip only; no cross-clip re-identification; no GT-derived identities',
                    relation_semantics='geometric association, not proof of manipulation',
                    format='wide object-centric CSV plus object inventory and relation table; not OCEL 2.0 serialization',
                    object_types=['person'] + sorted(object_types))
    (output_dir / 'event_log_metadata.json').write_text(json.dumps(metadata, indent=2) + '\n')
    return metadata


## Evaluation finale: metriche ed event log insieme
Eseguire questa cella dopo quelle di preparazione oppure usare **Esegui tutto**.

Controlla ed elabora entrambi gli input, calcola metriche per frame e per trace, crea l’event log e salva un unico risultato su Drive. Se mancano associazioni o un input non è valido, non pubblica una cartella di risultati parziale.

Lo ZIP include riepilogo testuale, metriche CSV/JSON, sequenze, mapping, event log con colonne per tipo oggetto, inventario oggetti, relazioni evento–oggetto e manifest degli input. I timestamp degli eventi restano relativi al video. Il nome della cartella usa UTC soltanto per identificare l’esecuzione, non come orario delle registrazioni.


In [ ]:
# Unica esecuzione finale: event log + metriche + riepiloghi + archivio.
import tempfile
from datetime import datetime, timezone
from uuid import uuid4

OUTPUT_ROOT = OUTPUT_DIR
run_name = datetime.now(timezone.utc).strftime('run_%Y%m%d_%H%M%S_UTC_') + uuid4().hex[:8]
RUN_OUTPUT = OUTPUT_ROOT / run_name

# Preparazione in una cartella temporanea: si pubblica il risultato completo
# soltanto se entrambe le elaborazioni terminano correttamente.
with tempfile.TemporaryDirectory(prefix='ground_final_') as staging:
    staging = Path(staging)
    event_log_summary = export_object_event_log(PREDICTIONS_ZIP, ASSOCIATIONS_PATH, staging)
    main([str(PREDICTIONS_ZIP), '--ground-truth', str(GT_CSV), '--output', str(staging)])
    manifest = dict(run_id=run_name, predictions_zip=str(PREDICTIONS_ZIP),
                    ground_truth_csv=str(GT_CSV), associations_path=str(ASSOCIATIONS_PATH),
                    metrics='summary.json', event_log='object_centric_event_log.csv',
                    event_log_summary=event_log_summary)
    (staging / 'run_manifest.json').write_text(json.dumps(manifest, indent=2) + '\n')
    OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)
    shutil.copytree(staging, RUN_OUTPUT)

print('\n' + (RUN_OUTPUT / 'evaluation_results.txt').read_text())
summary = json.loads((RUN_OUTPUT / 'summary.json').read_text())
index = {row['average']: row for row in summary['averages']}
print('\nCONFRONTO SENZA DROP')
for mode in ('MACRO', 'MICRO', 'WEIGHTED'):
    full = index[f'{mode}_AVG']['f1']
    without = index[f'{mode}_AVG_NO_DROP']['f1']
    print(f'{mode:8s} F1: tutte {full:.2%} | senza Drop {without:.2%} | differenza {(without-full)*100:+.2f} punti')

print('\nTRACE COMPLETE')
with (RUN_OUTPUT / 'per_trace.csv').open() as stream:
    for row in csv.DictReader(stream):
        if row['subset'] == 'ALL':
            print(f"\n{row['video']} — {row['person']}")
            print('GT:   ' + ' → '.join(json.loads(row['gt_sequence'])))
            print('Pred: ' + ' → '.join(json.loads(row['predicted_sequence'])))

print('\nOBJECT-CENTRIC EVENT LOG')
print(json.dumps(event_log_summary, indent=2))
archive_path = shutil.make_archive(str(RUN_OUTPUT), 'zip', root_dir=RUN_OUTPUT)
print(f'Risultati completi: {RUN_OUTPUT}')
print(f'Archivio: {archive_path}')
if IN_COLAB:
    from google.colab import files
    files.download(archive_path)
